# Wind tutorial

In [ ]:
import enn554.wind as wind
import matplotlib.pyplot as plt
from enn554.paths import data_dir
import numpy as np
import pandas as pd
dd = data_dir()

## Exercise 1

In [ ]:
coopers_gap = wind.merra_wind_speed_data()
coopers_gap.import_data(dd/"tutorial_4/POWER_Point_Hourly_20150101_20241231_026d73S_151d47E_UTC.csv")
coopers_gap.utc_to_lst(10) # we only really need to do this if we are aligning with the load.
coopers_gap.data

In [ ]:
ax = wind.wind_rose(coopers_gap.data['WD50M'],coopers_gap.data['WS50M'])
ax.set_title("Coopers Gap Wind Rose at 50m")

In [ ]:
coopers_gap.add_speed_at_height(100,alpha=0.14)
ax = wind.wind_rose(coopers_gap.data['WD100M'],coopers_gap.data['WS100M'],num_bins=16)
ax.set_title("Coopers Gap Wind Rose at 100m")

mean_speed_100 = coopers_gap.data['WS100M'].mean()
mean_direction_100 = wind.mean_direction(coopers_gap.data['WD100M'])
print(f"Mean speed: {mean_speed_100:.2f} m/s")
print(f"Mean direction: {mean_direction_100:.2f}° (meterological convention, clockwise from north)")

Fit the weibull distribution to the 100m data, with 16 azimuth bins.

In [ ]:
dist = wind.speed_fit(coopers_gap.data['WS100M'],plot=False)

fig, ax = plt.subplots()
x = np.linspace(0, coopers_gap.data['WS100M'].max(), 100)
ax.hist(coopers_gap.data['WS100M'],bins=20,density=True,label='Data')
ax.plot(x, dist.pdf(x), label='Fitted Weibull PDF', color='orange')
ax.set_xlabel("Wind Speed (m/s)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Wind Speed at 100m")
ax.axvline(mean_speed_100,color='red',linestyle='--',label=f"Mean: {mean_speed_100:.2f} m/s")
ax.legend()

Fit direction-dependent Weibull, which has a mixture PDF of the form
$$
    f(U,\theta) = \sum_{d=1}^D p_d \cdot f_d(v;c_d,k_d)
$$
where $p_d$ represnets the probability of the direction and $f_d$ is a Weibull pdf. 

In [ ]:
model = wind.speed_and_direction_dist(type='weibull',n_az_bins=16)
model.fit(coopers_gap.data['WD100M'],coopers_gap.data['WS100M'],plot=False)
model.print_params()

In [ ]:
T_sea_level =  15 #°C
T = T_sea_level - 0.66 + 273.15 # -0.66 K/100m
ρ_air = 101.3e3/287.1/T #kg/m3, using ideal gas law at 15°C and 101.3 kPa
WPD = 0.5*ρ_air*model.mean_speed()**3
print(f"Mean wind power density: {WPD:.2f} W/m²")

Velocity duration curve

In [ ]:
sorted_ws = coopers_gap.data['WS100M'].values
sorted_ws = np.flip(np.sort(sorted_ws))
hours_above = np.arange(1, len(sorted_ws)+1)
fig, ax = plt.subplots()
ax.plot(8760*hours_above/len(sorted_ws),sorted_ws) # scale x-axis to hours per year
ax.set_xlabel("Hours per year")
ax.set_ylabel("Wind Speed (m/s)")
ax.set_title("Speed duration curve at 100m")

## Exercise 2

Seimens first

In [ ]:
siemens_dat = pd.read_csv(dd/"tutorial_4/siemens.csv")
siemens = wind.turbine()
siemens.cut_in_speed = 3.0
siemens.cut_out_speed = 25.0
siemens.rated_power = 3.6e6
siemens.rotor_diameter  = 120
siemens.hub_height = 90
siemens.set_performance(siemens_dat['wind_speed_ms'].values,siemens_dat['power_kW'].values*1000)
siemens.plot()

Compute power duration curve

In [ ]:
powers_kW = np.array([siemens.get_power(ws,power_units='kW') for ws in coopers_gap.data['WS100M']])
sorted_power_kW = np.flip(np.sort(powers_kW))
hours_above = np.arange(1, len(sorted_power_kW)+1)
fig, ax = plt.subplots()
ax.plot(8760*hours_above/len(sorted_power_kW),sorted_power_kW) # scale x-axis to hours per year
ax.set_xlabel("Hours per year")
ax.set_ylabel("Power (kW)")
ax.set_title("Speed duration curve at 100m")

Weibull and AEP

In [ ]:
coopers_gap.add_speed_at_height([siemens.hub_height],alpha=0.14)
dist_siemens = wind.speed_fit(coopers_gap.data[f'WS{siemens.hub_height}M'],plot=False)
print(f"k={dist_siemens.args[0]:.2f}, c={dist_siemens.kwds['scale']:.2f}")
AEP_siemens = siemens.get_aep(dist_siemens,ref_height=siemens.hub_height,alpha=0.14,units='kW',dist_type='weibull',wind_profile='power_law')
print(f"Estimated AEP for siemens turbine: {AEP_siemens/1e6:.2f} GWh/year")

In [ ]:
siemens_dat = pd.read_csv(dd/"tutorial_4/siemens.csv")
siemens = wind.turbine()
siemens.cut_in_speed = 3.0
siemens.cut_out_speed = 25.0
siemens.rated_power = 3.6e6
siemens.rotor_diameter  = 120
siemens.hub_height = 90
siemens.set_performance(siemens_dat['wind_speed_ms'].values,siemens_dat['power_kW'].values*1000)
siemens.plot()

In [ ]:
coopers_gap.export_to_sam_csv(utc_offset=10,file_name=dd/"tutorial_4/coopers_gap_sam.csv",
                              date_range=("2023-01-01 00:00:00","2023-12-31 23:00:00"))

Repeat for ge one. 